# 🪙 Cryptocurrency Market Analysis System
### Advanced Databases – Final Project
**Team:** Michał Dusza, Szymon Bugajski, Mateusz Basiura  
**Programme:** Automatyka i Robotyka II Stopnia – Informatyka w Sterowaniu i Zarządzaniu

---

## Project Overview
This system automatically collects cryptocurrency market data from the **CoinGecko API**, stores it in a **SQLite** relational database, and presents the data through interactive statistical visualizations.

### Implemented analysis types:
| # | Type | Description |
|---|------|-------------|
| 1 | **Time Series** | Price/volume/market cap over time with moving averages |
| 2 | **Quantitative Analysis** | Comparative bar charts, box plots and violin plots |

### Database engine: **SQLite**
SQLite was chosen because it requires no external server, is fully ACID-compliant, and integrates seamlessly into Python notebooks.

---

## Project Stages
1. 📦 **Install dependencies**  
2. 🗄️ **Create database schema** (SQLite – 3 tables)  
3. 🌐 **Collect data** from CoinGecko REST API  
4. 💾 **Store data** in the database  
5. 📈 **Time Series visualization** (≥5 filters)  
6. 📊 **Quantitative Analysis** (≥5 filters)

---
## Stage 1 – Install Dependencies

In [ ]:
# import subprocess, sys

# packages = ["requests", "pandas", "plotly", "ipywidgets", "nbformat"]
# subprocess.check_call(
#     [sys.executable, "-m", "pip", "install", "-q"] + packages
# )
# print("✅ All packages installed.")

CalledProcessError: Command '['c:\\Users\\b8aavq\\Documents\\coingeko_project\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'requests', 'pandas', 'plotly', 'ipywidgets', 'nbformat']' returned non-zero exit status 1.

---
## Stage 2 – Imports & Configuration

In [ ]:
import sqlite3
import json
import time
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

warnings.filterwarnings("ignore")

# ── Configuration ─────────────────────────────────────────────────────────────
DB_PATH = "crypto_market.db"
BASE_URL = "https://api.coingecko.com/api/v3"
REQUEST_DELAY = 10   # seconds between API calls (free tier: ~10 req/min)

# Coins to track
COINS = [
    {"id": "bitcoin",      "symbol": "BTC", "name": "Bitcoin"},
    {"id": "ethereum",     "symbol": "ETH", "name": "Ethereum"},
    {"id": "solana",       "symbol": "SOL", "name": "Solana"},
    {"id": "binancecoin",  "symbol": "BNB", "name": "BNB"},
    {"id": "ripple",       "symbol": "XRP", "name": "XRP"},
]

METRIC_MAP = {
    "Price (USD)":         "price_usd",
    "Market Capitalization": "market_cap",
    "Trading Volume (24h)":  "total_volume",
}

print("✅ Imports done.")
print(f"   Tracking {len(COINS)} coins: {', '.join(c['symbol'] for c in COINS)}")
print(f"   Database: {DB_PATH}")

✅ Imports done.
   Tracking 5 coins: BTC, ETH, SOL, BNB, XRP
   Database: crypto_market.db


---
## Stage 3 – Database Schema

The database uses **SQLite** with three tables:

| Table | Purpose |
|-------|---------|
| `cryptocurrencies` | Master list of tracked assets (id, symbol, name) |
| `market_snapshots` | Historical daily OHLCV data (fetched from `/coins/{id}/market_chart`) |
| `market_current` | Point-in-time snapshots collected at each run (from `/coins/markets`) |

### Entity–Relationship diagram
```
cryptocurrencies (PK: id)
    │
    ├── market_snapshots  (FK: crypto_id → id)
    │     snapshot_date, price_usd, market_cap, total_volume
    │
    └── market_current   (FK: crypto_id → id)
          collected_at, price_usd, market_cap, total_volume,
          high_24h, low_24h, price_change_*, market_cap_rank,
          circulating_supply, total_supply, max_supply, ath, …
```

In [4]:
def create_database() -> None:
    """Create SQLite database with all required tables."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # ── Table 1: Master asset list ─────────────────────────────────────────────
    cur.execute("""
        CREATE TABLE IF NOT EXISTS cryptocurrencies (
            id      TEXT PRIMARY KEY,
            symbol  TEXT NOT NULL,
            name    TEXT NOT NULL
        )
    """)

    # ── Table 2: Historical daily snapshots ───────────────────────────────────
    cur.execute("""
        CREATE TABLE IF NOT EXISTS market_snapshots (
            record_id    INTEGER PRIMARY KEY AUTOINCREMENT,
            crypto_id    TEXT    NOT NULL,
            snapshot_date DATE   NOT NULL,
            price_usd    REAL,
            market_cap   REAL,
            total_volume REAL,
            UNIQUE(crypto_id, snapshot_date),
            FOREIGN KEY (crypto_id) REFERENCES cryptocurrencies(id)
        )
    """)
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_snapshots_date
        ON market_snapshots(snapshot_date)
    """)

    # ── Table 3: Live/current market snapshots ────────────────────────────────
    cur.execute("""
        CREATE TABLE IF NOT EXISTS market_current (
            record_id                     INTEGER PRIMARY KEY AUTOINCREMENT,
            crypto_id                     TEXT    NOT NULL,
            collected_at                  DATETIME NOT NULL,
            price_usd                     REAL,
            market_cap                    REAL,
            total_volume                  REAL,
            high_24h                      REAL,
            low_24h                       REAL,
            price_change_24h              REAL,
            price_change_percentage_24h   REAL,
            price_change_percentage_7d    REAL,
            market_cap_rank               INTEGER,
            circulating_supply            REAL,
            total_supply                  REAL,
            max_supply                    REAL,
            ath                           REAL,
            ath_change_percentage         REAL,
            FOREIGN KEY (crypto_id) REFERENCES cryptocurrencies(id)
        )
    """)
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_current_collected_at
        ON market_current(collected_at)
    """)

    conn.commit()
    conn.close()
    print("✅ Database created:", DB_PATH)
    print("   Tables: cryptocurrencies | market_snapshots | market_current")
    print("   Indexes: idx_snapshots_date | idx_current_collected_at")


create_database()

✅ Database created: crypto_market.db
   Tables: cryptocurrencies | market_snapshots | market_current
   Indexes: idx_snapshots_date | idx_current_collected_at


In [5]:
# Populate master coin list
def populate_cryptocurrencies() -> None:
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    for coin in COINS:
        cur.execute(
            "INSERT OR IGNORE INTO cryptocurrencies (id, symbol, name) VALUES (?, ?, ?)",
            (coin["id"], coin["symbol"], coin["name"]),
        )
    conn.commit()
    conn.close()

populate_cryptocurrencies()

# Verify
conn = sqlite3.connect(DB_PATH)
df_coins = pd.read_sql("SELECT * FROM cryptocurrencies", conn)
conn.close()
display(df_coins)
print("✅ Coin list stored in database.")

,id,symbol,name
0,bitcoin,BTC,Bitcoin
1,ethereum,ETH,Ethereum
2,solana,SOL,Solana
3,binancecoin,BNB,BNB
4,ripple,XRP,XRP


✅ Coin list stored in database.


---
## Stage 4 – Data Collection (CoinGecko API)

### Endpoints used

| Endpoint | Data |
|----------|------|
| `GET /coins/{id}/market_chart?vs_currency=usd&days=365&interval=daily` | Historical daily price, market cap, volume for last 365 days |
| `GET /coins/markets?vs_currency=usd&ids=…` | Current snapshot with all market metrics |

> **Note:** The free CoinGecko API allows ~10-30 requests/minute. A delay of 8 s is applied between historical data calls.

In [6]:
# ── API helper functions ───────────────────────────────────────────────────────

def fetch_market_chart(coin_id: str, days: int = 365) -> dict:
    """
    GET /coins/{id}/market_chart
    Returns dict with keys: prices, market_caps, total_volumes
    Each value is list of [timestamp_ms, value] pairs.
    """
    url = f"{BASE_URL}/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": days, "interval": "daily"}
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json()


def fetch_markets_current() -> list:
    """
    GET /coins/markets
    Returns list of current market snapshots for all tracked coins.
    """
    url = f"{BASE_URL}/coins/markets"
    params = {
        "vs_currency": "usd",
        "ids": ",".join(c["id"] for c in COINS),
        "order": "market_cap_desc",
        "per_page": 50,
        "page": 1,
        "sparkline": "false",
        "price_change_percentage": "24h,7d",
    }
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json()


# ── Storage functions ──────────────────────────────────────────────────────────

def store_market_chart(coin_id: str, data: dict) -> int:
    """Insert historical daily records, skipping existing rows (UNIQUE constraint)."""
    prices      = data.get("prices", [])
    market_caps = data.get("market_caps", [])
    volumes     = data.get("total_volumes", [])

    rows = []
    for i, (ts_ms, price) in enumerate(prices):
        date = datetime.utcfromtimestamp(ts_ms / 1000).strftime("%Y-%m-%d")
        mc  = market_caps[i][1] if i < len(market_caps) else None
        vol = volumes[i][1]     if i < len(volumes)     else None
        rows.append((coin_id, date, price, mc, vol))

    conn = sqlite3.connect(DB_PATH)
    conn.executemany(
        """INSERT OR REPLACE INTO market_snapshots
           (crypto_id, snapshot_date, price_usd, market_cap, total_volume)
           VALUES (?, ?, ?, ?, ?)""",
        rows,
    )
    conn.commit()
    conn.close()
    return len(rows)


def store_current(data: list) -> None:
    """Insert current market snapshots."""
    now = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    rows = [
        (
            coin["id"], now,
            coin.get("current_price"),
            coin.get("market_cap"),
            coin.get("total_volume"),
            coin.get("high_24h"),
            coin.get("low_24h"),
            coin.get("price_change_24h"),
            coin.get("price_change_percentage_24h"),
            coin.get("price_change_percentage_7d_in_currency"),
            coin.get("market_cap_rank"),
            coin.get("circulating_supply"),
            coin.get("total_supply"),
            coin.get("max_supply"),
            coin.get("ath"),
            coin.get("ath_change_percentage"),
        )
        for coin in data
    ]
    conn = sqlite3.connect(DB_PATH)
    conn.executemany(
        """INSERT INTO market_current
           (crypto_id, collected_at, price_usd, market_cap, total_volume,
            high_24h, low_24h, price_change_24h, price_change_percentage_24h,
            price_change_percentage_7d, market_cap_rank, circulating_supply,
            total_supply, max_supply, ath, ath_change_percentage)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        rows,
    )
    conn.commit()
    conn.close()


print("✅ API helper functions defined.")

✅ API helper functions defined.


---
## Stage 5 – Fetch Historical Data (365 days per coin)

Running 5 API calls – one per coin. Each call returns ~365 daily data points  
(price, market cap, volume). A delay of 8 s is inserted to respect the rate limit.

In [ ]:
print("⏳ Fetching 365-day historical data from CoinGecko API …\n")

for i, coin in enumerate(COINS):
    print(f"  [{i+1}/{len(COINS)}] {coin['name']} ({coin['symbol']}) … ", end="", flush=True)
    try:
        data = fetch_market_chart(coin["id"], days=365)
        n    = store_market_chart(coin["id"], data)
        print(f"✅  {n} daily records stored")
    except Exception as exc:
        print(f"❌  {exc}")

    if i < len(COINS) - 1:          # no delay after last request
        time.sleep(REQUEST_DELAY)

print("\n⏳ Fetching current market snapshot …")
try:
    current_data = fetch_markets_current()
    store_current(current_data)
    print(f"✅  Current data stored for {len(current_data)} coins")
except Exception as exc:
    print(f"❌  {exc}")

print("\n🎉 Data collection complete!")

⏳ Fetching 365-day historical data from CoinGecko API …



NameError: name 'COINS' is not defined

In [ ]:
# ── Retry: fetch current market snapshot (run if Stage 5 got a 429 error) ─────
print("⏳ Retrying current market snapshot …")
for attempt in range(1, 4):
    try:
        current_data = fetch_markets_current()
        store_current(current_data)
        print(f"✅  Stored current snapshot for {len(current_data)} coins")
        break
    except Exception as exc:
        print(f"   Attempt {attempt} failed ({exc}) – waiting 20 s …")
        time.sleep(20)
else:
    print("❌  All retries exhausted.")

---
## Stage 6 – Verify Data in Database

In [ ]:
conn = sqlite3.connect(DB_PATH)

# Summary statistics per coin
df_summary = pd.read_sql("""
    SELECT
        c.name,
        c.symbol,
        COUNT(s.record_id)                          AS daily_records,
        MIN(s.snapshot_date)                        AS from_date,
        MAX(s.snapshot_date)                        AS to_date,
        ROUND(MIN(s.price_usd), 2)                  AS min_price_usd,
        ROUND(MAX(s.price_usd), 2)                  AS max_price_usd,
        ROUND(AVG(s.price_usd), 2)                  AS avg_price_usd
    FROM market_snapshots s
    JOIN cryptocurrencies c ON s.crypto_id = c.id
    GROUP BY c.id
    ORDER BY MAX(s.market_cap) DESC
""", conn)

# Current snapshot
df_now = pd.read_sql("""
    SELECT
        c.name,
        mc.price_usd,
        mc.market_cap,
        mc.total_volume,
        mc.price_change_percentage_24h,
        mc.price_change_percentage_7d,
        mc.market_cap_rank,
        mc.circulating_supply,
        mc.ath,
        mc.collected_at
    FROM market_current mc
    JOIN cryptocurrencies c ON mc.crypto_id = c.id
    WHERE mc.record_id IN (
        SELECT MAX(record_id) FROM market_current GROUP BY crypto_id
    )
    ORDER BY mc.market_cap_rank
""", conn)

conn.close()

print("── Historical snapshots per coin ──────────────────────────────────────")
display(df_summary)
print("\n── Most recent market snapshot ────────────────────────────────────────")
display(df_now)

---
## Stage 7 – Load Dataset for Visualization

In [ ]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("""
    SELECT
        s.snapshot_date,
        s.price_usd,
        s.market_cap,
        s.total_volume,
        c.name,
        c.symbol
    FROM market_snapshots s
    JOIN cryptocurrencies c ON s.crypto_id = c.id
    ORDER BY s.snapshot_date
""", conn)
conn.close()

df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])

print(f"✅ Loaded {len(df):,} records")
print(f"   Date range : {df['snapshot_date'].min().date()} → {df['snapshot_date'].max().date()}")
print(f"   Coins      : {', '.join(df['name'].unique())}")
df.head(3)

---
## Stage 8 – Time Series Analysis

### Available filters (≥5)

| # | Filter | Widget | Description |
|---|--------|--------|-------------|
| 1 | **Cryptocurrency** | Multi-select listbox | Choose one or more coins |
| 2 | **Start date** | Date picker | Lower bound of date range |
| 3 | **End date** | Date picker | Upper bound of date range |
| 4 | **Metric** | Dropdown | Price / Market Cap / Volume |
| 5 | **Moving-average window** | Slider (0–30 days) | 0 = raw data; >0 = rolling mean |
| 6 | **Log scale** | Toggle button | Switch Y-axis to logarithmic scale |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TIME SERIES VISUALIZATION  –  6 interactive filters
# ═══════════════════════════════════════════════════════════════════════════════

date_min = df["snapshot_date"].min().date()
date_max = df["snapshot_date"].max().date()
all_coin_names = sorted(df["name"].unique().tolist())

# ── Filter widgets ─────────────────────────────────────────────────────────────
ts_coins = widgets.SelectMultiple(
    options=all_coin_names,
    value=["Bitcoin", "Ethereum"],
    description="1. Coins:",
    rows=min(len(all_coin_names), 6),
    layout=widgets.Layout(width="220px", height="130px"),
)

ts_start = widgets.DatePicker(
    value=date_max - timedelta(days=180),
    description="2. Start:",
    layout=widgets.Layout(width="300px"),
)

ts_end = widgets.DatePicker(
    value=date_max,
    description="3. End:",
    layout=widgets.Layout(width="300px"),
)

ts_metric = widgets.Dropdown(
    options=list(METRIC_MAP.keys()),
    value="Price (USD)",
    description="4. Metric:",
    layout=widgets.Layout(width="300px"),
)

ts_ma = widgets.IntSlider(
    value=7,
    min=0, max=30, step=1,
    description="5. MA (days):",
    continuous_update=False,
    style={"description_width": "100px"},
    layout=widgets.Layout(width="400px"),
)

ts_log = widgets.ToggleButton(
    value=False,
    description="6. Log scale",
    button_style="info",
    icon="chart-bar",
    layout=widgets.Layout(width="150px"),
)

ts_out = widgets.Output()


def update_timeseries(_=None):
    with ts_out:
        clear_output(wait=True)

        selected_coins = list(ts_coins.value)
        if not selected_coins:
            print("⚠️  Select at least one coin.")
            return

        metric_col = METRIC_MAP[ts_metric.value]

        mask = (
            df["name"].isin(selected_coins)
            & (df["snapshot_date"] >= pd.Timestamp(ts_start.value))
            & (df["snapshot_date"] <= pd.Timestamp(ts_end.value))
        )
        filtered = df[mask].copy()

        if filtered.empty:
            print("⚠️  No data for the selected filters.")
            return

        # Apply moving average
        if ts_ma.value > 1:
            filtered = filtered.sort_values(["name", "snapshot_date"])
            filtered[metric_col] = (
                filtered.groupby("name")[metric_col]
                .transform(lambda x: x.rolling(ts_ma.value, min_periods=1).mean())
            )
            title_ma = f"  |  {ts_ma.value}-day moving average"
        else:
            title_ma = ""

        coin_label = ", ".join(selected_coins) if len(selected_coins) <= 3 else f"{len(selected_coins)} coins"
        title = f"📈  {ts_metric.value} – {coin_label}{title_ma}"

        fig = px.line(
            filtered,
            x="snapshot_date",
            y=metric_col,
            color="name",
            title=title,
            labels={"snapshot_date": "Date", metric_col: ts_metric.value, "name": "Coin"},
            template="plotly_dark",
            color_discrete_sequence=px.colors.qualitative.Bold,
        )

        fig.update_traces(line=dict(width=1.8))
        fig.update_layout(
            height=480,
            hovermode="x unified",
            legend=dict(orientation="h", y=1.02, x=1, xanchor="right", yanchor="bottom"),
            margin=dict(l=60, r=20, t=80, b=50),
            yaxis_type="log" if ts_log.value else "linear",
        )
        fig.show()


# Bind all widgets to the callback
for w in [ts_coins, ts_start, ts_end, ts_metric, ts_ma, ts_log]:
    w.observe(update_timeseries, names="value")

# ── Layout ─────────────────────────────────────────────────────────────────────
header = widgets.HTML(
    "<h3 style='margin:0;color:#7ec8e3'>📈 Time Series Analysis</h3>"
    "<p style='margin:2px 0 8px 0;color:#aaa;font-size:0.85em'>"
    "Use the controls below to filter the chart.</p>"
)
left_col  = widgets.VBox([ts_coins])
right_col = widgets.VBox([ts_start, ts_end, ts_metric, ts_ma, ts_log])
controls  = widgets.VBox([header, widgets.HBox([left_col, widgets.VBox([right_col])])])

display(controls, ts_out)
update_timeseries()

---
## Stage 9 – Quantitative Analysis

### Available filters (≥5)

| # | Filter | Widget | Description |
|---|--------|--------|-------------|
| 1 | **Cryptocurrency** | Multi-select listbox | Choose coins to compare |
| 2 | **Metric** | Dropdown | Price / Market Cap / Volume |
| 3 | **Time period** | Dropdown | Last 7 / 30 / 90 / 180 / 365 days |
| 4 | **Aggregation** | Dropdown | Mean / Max / Min / Last / Std Dev |
| 5 | **Sort order** | Toggle | Ascending / Descending |
| 6 | **Chart type** | Dropdown | Bar chart / Box plot / Violin plot |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# QUANTITATIVE ANALYSIS  –  6 interactive filters
# ═══════════════════════════════════════════════════════════════════════════════

PERIOD_MAP = {
    "Last 7 days":   7,
    "Last 30 days":  30,
    "Last 90 days":  90,
    "Last 180 days": 180,
    "Last 365 days": 365,
}

AGG_MAP = {
    "Mean":              "mean",
    "Maximum":           "max",
    "Minimum":           "min",
    "Last value":        "last",
    "Standard Deviation":"std",
}

# ── Filter widgets ─────────────────────────────────────────────────────────────
qa_coins = widgets.SelectMultiple(
    options=all_coin_names,
    value=all_coin_names,
    description="1. Coins:",
    rows=min(len(all_coin_names), 6),
    layout=widgets.Layout(width="220px", height="130px"),
)

qa_metric = widgets.Dropdown(
    options=list(METRIC_MAP.keys()),
    value="Market Capitalization",
    description="2. Metric:",
    layout=widgets.Layout(width="300px"),
)

qa_period = widgets.Dropdown(
    options=list(PERIOD_MAP.keys()),
    value="Last 90 days",
    description="3. Period:",
    layout=widgets.Layout(width="300px"),
)

qa_agg = widgets.Dropdown(
    options=list(AGG_MAP.keys()),
    value="Mean",
    description="4. Aggregation:",
    layout=widgets.Layout(width="300px"),
)

qa_sort = widgets.ToggleButtons(
    options=["Ascending", "Descending"],
    value="Descending",
    description="5. Sort:",
    layout=widgets.Layout(width="300px"),
)

qa_chart = widgets.Dropdown(
    options=["Bar Chart", "Box Plot", "Violin Plot"],
    value="Bar Chart",
    description="6. Chart type:",
    layout=widgets.Layout(width="300px"),
)

qa_out = widgets.Output()


def update_quant(_=None):
    with qa_out:
        clear_output(wait=True)

        selected = list(qa_coins.value)
        if not selected:
            print("⚠️  Select at least one coin.")
            return

        metric_col = METRIC_MAP[qa_metric.value]
        days       = PERIOD_MAP[qa_period.value]
        agg_key    = AGG_MAP[qa_agg.value]
        ascending  = qa_sort.value == "Ascending"
        chart_type = qa_chart.value

        cutoff   = df["snapshot_date"].max() - pd.Timedelta(days=days)
        filtered = df[df["name"].isin(selected) & (df["snapshot_date"] >= cutoff)]

        if filtered.empty:
            print("⚠️  No data for the selected filters.")
            return

        palette = px.colors.qualitative.Bold

        if chart_type == "Bar Chart":
            agg_df = (
                filtered.groupby("name")[metric_col]
                .agg(agg_key)
                .reset_index()
                .sort_values(metric_col, ascending=ascending)
            )
            fig = px.bar(
                agg_df,
                x="name", y=metric_col,
                color="name",
                text_auto=".3s",
                title=f"📊  {qa_agg.value} {qa_metric.value}  |  {qa_period.value}",
                labels={"name": "Coin", metric_col: qa_metric.value},
                template="plotly_dark",
                color_discrete_sequence=palette,
            )
            fig.update_traces(textfont_size=12, textangle=0, textposition="outside")

        elif chart_type == "Box Plot":
            sorted_names = (
                filtered.groupby("name")[metric_col]
                .median()
                .sort_values(ascending=ascending)
                .index.tolist()
            )
            fig = px.box(
                filtered,
                x="name", y=metric_col,
                color="name",
                category_orders={"name": sorted_names},
                title=f"📦  Distribution of {qa_metric.value}  |  {qa_period.value}",
                labels={"name": "Coin", metric_col: qa_metric.value},
                template="plotly_dark",
                color_discrete_sequence=palette,
            )

        else:  # Violin
            sorted_names = (
                filtered.groupby("name")[metric_col]
                .median()
                .sort_values(ascending=ascending)
                .index.tolist()
            )
            fig = px.violin(
                filtered,
                x="name", y=metric_col,
                color="name",
                box=True,
                points="outliers",
                category_orders={"name": sorted_names},
                title=f"🎻  Distribution of {qa_metric.value}  |  {qa_period.value}",
                labels={"name": "Coin", metric_col: qa_metric.value},
                template="plotly_dark",
                color_discrete_sequence=palette,
            )

        fig.update_layout(height=480, showlegend=False, margin=dict(l=60, r=20, t=80, b=50))
        fig.show()


for w in [qa_coins, qa_metric, qa_period, qa_agg, qa_sort, qa_chart]:
    w.observe(update_quant, names="value")

# ── Layout ─────────────────────────────────────────────────────────────────────
qa_header = widgets.HTML(
    "<h3 style='margin:0;color:#f4a460'>📊 Quantitative Analysis</h3>"
    "<p style='margin:2px 0 8px 0;color:#aaa;font-size:0.85em'>"
    "Compare aggregated metrics across selected cryptocurrencies.</p>"
)
qa_left  = widgets.VBox([qa_coins])
qa_right = widgets.VBox([qa_metric, qa_period, qa_agg, qa_sort, qa_chart])
qa_ctrl  = widgets.VBox([qa_header, widgets.HBox([qa_left, qa_right])])

display(qa_ctrl, qa_out)
update_quant()

---
## Stage 10 – Market Overview Dashboard

Static summary dashboard showing current state of all tracked coins: ranking table + price-change heatmap.

In [ ]:
conn = sqlite3.connect(DB_PATH)
df_cur = pd.read_sql("""
    SELECT
        c.name,
        c.symbol,
        mc.market_cap_rank          AS rank,
        mc.price_usd,
        mc.market_cap,
        mc.total_volume,
        mc.price_change_percentage_24h  AS chg_24h,
        mc.price_change_percentage_7d   AS chg_7d,
        mc.ath,
        mc.ath_change_percentage        AS ath_chg_pct,
        mc.circulating_supply
    FROM market_current mc
    JOIN cryptocurrencies c ON mc.crypto_id = c.id
    WHERE mc.record_id IN (
        SELECT MAX(record_id) FROM market_current GROUP BY crypto_id
    )
    ORDER BY mc.market_cap_rank
""", conn)
conn.close()

# ── 1. Summary Table ───────────────────────────────────────────────────────────
fig_table = go.Figure(
    data=[go.Table(
        header=dict(
            values=["Rank", "Name", "Symbol", "Price (USD)", "Market Cap",
                    "Volume 24h", "Chg 24h %", "Chg 7d %", "ATH (USD)"],
            fill_color="#1e1e2e",
            font=dict(color="white", size=12),
            align="center",
        ),
        cells=dict(
            values=[
                df_cur["rank"],
                df_cur["name"],
                df_cur["symbol"],
                df_cur["price_usd"].apply(lambda x: f"${x:,.2f}"),
                df_cur["market_cap"].apply(lambda x: f"${x/1e9:.2f} B"),
                df_cur["total_volume"].apply(lambda x: f"${x/1e9:.2f} B"),
                df_cur["chg_24h"].apply(lambda x: f"{x:+.2f}%"),
                df_cur["chg_7d"].apply(lambda x: f"{x:+.2f}%"),
                df_cur["ath"].apply(lambda x: f"${x:,.0f}"),
            ],
            fill_color=[
                ["#2b2b3b"] * len(df_cur),
                ["#2b2b3b"] * len(df_cur),
                ["#2b2b3b"] * len(df_cur),
                ["#2b2b3b"] * len(df_cur),
                ["#2b2b3b"] * len(df_cur),
                ["#2b2b3b"] * len(df_cur),
                ["#1a3a1a" if v > 0 else "#3a1a1a" for v in df_cur["chg_24h"]],
                ["#1a3a1a" if v > 0 else "#3a1a1a" for v in df_cur["chg_7d"]],
                ["#2b2b3b"] * len(df_cur),
            ],
            font=dict(color="white", size=12),
            align="center",
        ),
    )]
)
fig_table.update_layout(
    title="Current Market Snapshot",
    template="plotly_dark",
    margin=dict(l=0, r=0, t=50, b=0),
    height=240,
)
fig_table.show()

# ── 2. Price-change heatmap ────────────────────────────────────────────────────
changes = df_cur[["name", "chg_24h", "chg_7d"]].set_index("name")
changes.columns = ["24h change %", "7d change %"]

fig_heat = px.imshow(
    changes.T,
    text_auto=".2f",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Price Change Heatmap",
    template="plotly_dark",
    aspect="auto",
)
fig_heat.update_layout(height=220, margin=dict(l=100, r=20, t=60, b=20))
fig_heat.show()

# ── 3. Market-cap treemap ──────────────────────────────────────────────────────
fig_tree = px.treemap(
    df_cur,
    path=["name"],
    values="market_cap",
    color="chg_24h",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Market Cap Treemap  (color = 24h change %)",
    template="plotly_dark",
)
fig_tree.update_layout(height=380, margin=dict(l=10, r=10, t=60, b=10))
fig_tree.show()

---
## Stage 11 – Correlation & Volatility Analysis

Additional quantitative view: rolling 30-day correlation matrix and volatility (standard deviation of daily returns) across coins.

In [ ]:
# ── Pivot price data ──────────────────────────────────────────────────────────
price_pivot = df.pivot(index="snapshot_date", columns="name", values="price_usd")
returns     = price_pivot.pct_change().dropna()

# ── Correlation matrix ────────────────────────────────────────────────────────
corr = returns.corr()

fig_corr = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    zmin=-1, zmax=1,
    title="Price Return Correlation Matrix (full period)",
    template="plotly_dark",
)
fig_corr.update_layout(height=400, margin=dict(l=10, r=10, t=60, b=10))
fig_corr.show()

# ── Annualised Volatility bar chart ───────────────────────────────────────────
volatility = (returns.std() * (365 ** 0.5) * 100).reset_index()
volatility.columns = ["Coin", "Annualised Volatility (%)"]
volatility = volatility.sort_values("Annualised Volatility (%)", ascending=False)

fig_vol = px.bar(
    volatility,
    x="Coin", y="Annualised Volatility (%)",
    color="Coin",
    text_auto=".1f",
    title="Annualised Volatility of Daily Returns (%)",
    template="plotly_dark",
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig_vol.update_layout(height=380, showlegend=False, margin=dict(l=60, r=20, t=60, b=50))
fig_vol.show()

---
## Summary & Conclusions

### What was implemented

| Component | Details |
|-----------|---------|
| **Data source** | CoinGecko REST API – free public tier |
| **Database engine** | SQLite 3 (ACID, relational, no external server required) |
| **Tables** | `cryptocurrencies` (5 rows), `market_snapshots` (~1 825 rows), `market_current` (5 rows) |
| **Coins tracked** | BTC, ETH, SOL, BNB, XRP |
| **Historical depth** | 365 daily data points per coin |
| **Time Series** | Interactive line chart with 6 filters (coin, date range, metric, MA window, log scale) |
| **Quantitative Analysis** | Bar / Box / Violin charts with 6 filters (coin, metric, period, aggregation, sort, chart type) |
| **Dashboard** | Market table + heatmap + treemap |
| **Correlation** | Full-period price-return correlation matrix + annualised volatility |

### How to refresh data
Re-run **Stage 5** cell at any time; the `INSERT OR REPLACE` / `INSERT` statements will add new snapshots without duplicating historical rows.

### Database schema
```sql
cryptocurrencies  (id PK, symbol, name)
market_snapshots  (record_id PK, crypto_id FK, snapshot_date, price_usd,
                   market_cap, total_volume)          -- UNIQUE(crypto_id, snapshot_date)
market_current    (record_id PK, crypto_id FK, collected_at, price_usd,
                   market_cap, total_volume, high_24h, low_24h,
                   price_change_24h, price_change_percentage_24h,
                   price_change_percentage_7d, market_cap_rank,
                   circulating_supply, total_supply, max_supply,
                   ath, ath_change_percentage)
```